In [1]:
# Tracing: arize-otel. Evaluation logging: arize (provides arize.pandas.logger.Client for log_evaluations_sync). Pinned to v7 (v8 causes issues).
#!pip install -qqq arize-otel "arize>=7,<8"

In [2]:
import os
space_id = os.environ["ARIZE_SPACE_ID"]
api_key = os.environ["ARIZE_API_KEY"]

In [3]:
project_name = "findata_research_report_augment-6"

In [4]:
# Import open-telemetry dependencies
from arize.otel import register
import pandas as pd
import json

tracer_provider = register(
    space_id=space_id,
    api_key=api_key,
    project_name=project_name,
)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: findata_research_report_augment-6
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [5]:
# Vertex AI / Gemini metadata (for span attributes only; no real API calls)
# ADK-aligned: use Vertex model names for observability
VERTEX_LOCATION = "us-central1"
VERTEX_MODEL_ID = "gemini-2.0-flash"

In [6]:
# FinData Research Report Augment Data Configuration
# Analyst-style queries, research document catalog, first-pull, reranked list, report-ready response

import random

# Catalog: research reports / documents (id, title, sector, report_type, description)
CATALOG = [
    {"id": "r1", "title": "Tech Q3 Earnings Summary", "sector": "Technology", "report_type": "earnings", "description": "AAPL, MSFT, GOOGL consensus and revisions; cloud and AI drivers."},
    {"id": "r2", "title": "US Banks Margin Comparison", "sector": "Financials", "report_type": "sector", "description": "Top 5 US banks Q3 margins, NII pressure, trading revenue."},
    {"id": "r3", "title": "European Pharma M&A Update", "sector": "Healthcare", "report_type": "ma", "description": "Recent deals, deal sizes, strategic rationale."},
    {"id": "r4", "title": "Retail Earnings Themes", "sector": "Consumer", "report_type": "earnings", "description": "Key themes from latest retail earnings; margin pressure names."},
    {"id": "r5", "title": "Fed Pricing and Data Dependencies", "sector": "Macro", "report_type": "macro", "description": "Rates path, next two meetings, labor and inflation data."},
    {"id": "r6", "title": "AAPL FY25 Revenue Consensus", "sector": "Technology", "report_type": "earnings", "description": "Consensus revenue estimate for AAPL FY25 and 3-month revision history."},
    {"id": "r7", "title": "Bank NII and Credit Quality", "sector": "Financials", "report_type": "sector", "description": "NII trends, funding costs, credit quality across universal banks."},
    {"id": "r8", "title": "Biotech Approvals and M&A", "sector": "Healthcare", "report_type": "ma", "description": "FDA approvals, mid-cap biotech deal flow."},
    {"id": "r9", "title": "Consumer Discretionary and Housing", "sector": "Consumer", "report_type": "sector", "description": "Retail sales, housing affordability, existing home sales."},
    {"id": "r10", "title": "Dot Plot and Fed Communication", "sector": "Macro", "report_type": "macro", "description": "Dot plot implications, one cut by year-end baseline."},
]

# Per-query scenario: analyst question, first_pull (doc ids + scores), reranked (reordered), response (report-ready, cited)
SEARCH_AUGMENT_CONFIG = [
    {
        "query": "What's the consensus revenue estimate for AAPL in FY25 and how has it changed over the last three months?",
        "first_pull": [("r6", 0.90), ("r1", 0.82), ("r2", 0.45), ("r5", 0.40)],
        "reranked": [("r6", 0.96), ("r1", 0.88)],
        "response": "Per **AAPL FY25 Revenue Consensus**, consensus revenue for FY25 is $XXB, up from $YYB three months ago on stronger Services and iPhone. **Tech Q3 Earnings Summary** notes continued cloud and AI momentum supporting top-line revisions.",
    },
    {
        "query": "Compare Q3 margins for the top five US banks and highlight main drivers.",
        "first_pull": [("r2", 0.92), ("r7", 0.88), ("r1", 0.50), ("r5", 0.42)],
        "reranked": [("r2", 0.97), ("r7", 0.91)],
        "response": "**US Banks Margin Comparison** and **Bank NII and Credit Quality** show mixed Q3 margins across the top five; NII pressure from funding costs was partly offset by stronger trading revenue. Key drivers: deposit betas, loan growth, and trading volatility.",
    },
    {
        "query": "Summarize recent M&A activity in European pharma with deal sizes and strategic rationale.",
        "first_pull": [("r3", 0.94), ("r8", 0.85), ("r2", 0.48), ("r4", 0.44)],
        "reranked": [("r3", 0.98), ("r8", 0.90)],
        "response": "**European Pharma M&A Update** and **Biotech Approvals and M&A** summarize recent activity: several mid-cap deals in the $2–5B range, with strategic rationale focused on pipeline fill and therapeutic area expansion. Deal flow expected to pick up into year-end.",
    },
    {
        "query": "What are the key themes from the latest retail earnings and which names are most at risk from margin pressure?",
        "first_pull": [("r4", 0.91), ("r9", 0.80), ("r1", 0.55), ("r3", 0.48)],
        "reranked": [("r4", 0.95), ("r9", 0.87)],
        "response": "**Retail Earnings Themes** highlights strength in back-to-school and discretionary, with travel robust. Names most at risk from margin pressure include those with labor and inventory headwinds; **Consumer Discretionary and Housing** notes affordability and existing home sales softness.",
    },
    {
        "query": "Give me a concise view on Fed pricing for the next two meetings and the main data dependencies.",
        "first_pull": [("r5", 0.93), ("r10", 0.89), ("r2", 0.52), ("r1", 0.45)],
        "reranked": [("r5", 0.96), ("r10", 0.92)],
        "response": "**Fed Pricing and Data Dependencies** and **Dot Plot and Fed Communication** suggest one cut by year-end baseline; next two meetings are data-dependent. Key inputs: labor, inflation prints, and Fed communication. Dot plot implies a cautious path.",
    },
    {
        "query": "How have tech earnings revisions trended for cloud and AI exposure?",
        "first_pull": [("r1", 0.90), ("r6", 0.84), ("r4", 0.50), ("r5", 0.46)],
        "reranked": [("r1", 0.95), ("r6", 0.88)],
        "response": "**Tech Q3 Earnings Summary** indicates earnings beat estimates with cloud and AI driving growth; **AAPL FY25 Revenue Consensus** aligns with positive revision trends for names with strong cloud and AI exposure.",
    },
]

def _catalog_by_id():
    return {item["id"]: item for item in CATALOG}

def get_search_augment_data(query_id=None):
    """Get data for one research-report search-augment trace. query_id: 0..len(SEARCH_AUGMENT_CONFIG)-1 or None for random."""
    configs = SEARCH_AUGMENT_CONFIG
    if query_id is None:
        query_id = random.randint(0, len(configs) - 1)
    idx = query_id % len(configs)
    cfg = configs[idx]
    catalog_map = _catalog_by_id()
    retrieved = []
    for doc_id, score in cfg["first_pull"]:
        item = catalog_map.get(doc_id, {"id": doc_id, "title": doc_id, "sector": "", "report_type": "", "description": ""})
        content = f"{item['title']} ({item['sector']}, {item['report_type']}). {item['description']}"
        retrieved.append({"id": doc_id, "content": content, "score": round(score + random.uniform(-0.02, 0.02), 2)})
    reranked = []
    for doc_id, score in cfg["reranked"]:
        item = catalog_map.get(doc_id, {"id": doc_id, "title": doc_id, "sector": "", "report_type": "", "description": ""})
        content = f"{item['title']} ({item['sector']}, {item['report_type']}). {item['description']}"
        reranked.append({"id": doc_id, "content": content, "score": round(score + random.uniform(-0.01, 0.01), 2)})
    return {
        "query_id": idx,
        "query": cfg["query"],
        "retrieved_docs": retrieved,
        "reranked_docs": reranked,
        "response": cfg["response"],
    }

In [7]:
# Prompt template for LLM: synthesize_analyst_answer (query + reranked docs -> report-ready answer)
PROMPT_TEMPLATE_CONFIG = {
    "synthesize_analyst_answer": {
        "template": (
            "The analyst asked: \"{query}\"\n\n"
            "Reranked research documents (use only these):\n{reranked_list}\n\n"
            "Produce a short report-ready answer citing only these documents. Be concise and cite document titles where relevant."
        ),
        "system_message": (
            "You are a FinData research assistant. Given an analyst question and a reranked list of research documents, "
            "produce a short, report-ready answer. Use only information from the provided documents; cite document titles where relevant."
        ),
        "version": "v1.0",
    }
}

In [8]:
# Timer utility
from contextlib import contextmanager
import time

@contextmanager
def timer():
    start = time.time()
    yield
    print(f"Execution time: {time.time() - start:.2f} seconds")

In [9]:
# Trace: CHAIN -> RETRIEVER (candidate_retrieval) -> RERANKER (rerank_candidates) -> AGENT (research_report_agent) -> LLM (synthesize_analyst_answer)
# ADK-aligned: Vertex model, snake_case agent name; Arize: prompt template attributes on LLM span
from opentelemetry import trace
from opentelemetry.trace.status import Status, StatusCode
import json
import time
import random

def create_search_augment_trace(tracer, query_id=None, session_id=None):
    """Create one FinData research-report search-augment trace. Returns dict with span_data."""
    data = get_search_augment_data(query_id)
    query = data["query"]
    retrieved_docs = data["retrieved_docs"]
    reranked_docs = data["reranked_docs"]
    response = data["response"]

    cfg = PROMPT_TEMPLATE_CONFIG.get("synthesize_analyst_answer", {})
    reranked_list = "\n".join(f"- {d['id']}: {d['content'][:80]}..." for d in reranked_docs)
    user_msg = cfg.get("template", "").format(query=query, reranked_list=reranked_list)
    system_msg = cfg.get("system_message", "You are a helpful assistant.")

    # Prompt template variables for Arize Prompt Playground (truncate long values for span attribute)
    prompt_variables = {"query": query[:500], "reranked_list": reranked_list[:1500]}
    prompt_template_str = cfg.get("template", "")
    prompt_version = cfg.get("version", "v1.0")

    chain_name = "FinData Research Report Pipeline"
    retrieval_latency_ms = random.uniform(50, 200)
    reranker_latency_ms = random.uniform(80, 250)
    agent_latency_ms = random.uniform(30, 120)
    llm_latency_ms = random.uniform(800, 2500)
    chain_latency_ms = retrieval_latency_ms + reranker_latency_ms + agent_latency_ms + llm_latency_ms + random.uniform(20, 80)

    chain_span_id = retrieval_span_id = reranker_span_id = agent_span_id = llm_span_id = None

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", query)
        chain_span.set_attribute("input.mime_type", "text/plain")
        if session_id:
            chain_span.set_attribute("session.id", session_id)
        time.sleep(0.003)
        ctx = chain_span.get_span_context()
        chain_span_id = format(ctx.span_id, "016x")

        with tracer.start_as_current_span("candidate_retrieval") as ret_span:
            ret_span.set_attribute("openinference.span.kind", "RETRIEVER")
            ret_span.set_attribute("input.value", json.dumps({"query": query}))
            ret_span.set_attribute("input.mime_type", "application/json")
            for idx, doc in enumerate(retrieved_docs):
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.id", doc["id"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                ret_span.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
            ret_span.set_attribute("output.value", json.dumps([{"id": d["id"], "score": d["score"]} for d in retrieved_docs]))
            ret_span.set_attribute("output.mime_type", "application/json")
            ret_span.set_attribute("latency_ms", round(retrieval_latency_ms, 2))
            ret_span.set_status(Status(StatusCode.OK))
            ctx = ret_span.get_span_context()
            retrieval_span_id = format(ctx.span_id, "016x")

        with tracer.start_as_current_span("rerank_candidates") as rer_span:
            rer_span.set_attribute("openinference.span.kind", "RERANKER")
            rer_span.set_attribute("input.value", json.dumps({"query": query, "document_ids": [d["id"] for d in retrieved_docs], "scores": [d["score"] for d in retrieved_docs]}))
            rer_span.set_attribute("output.value", json.dumps([{"id": d["id"], "score": d["score"]} for d in reranked_docs]))
            rer_span.set_attribute("reranker.model_name", "cross-encoder-reranker-v1")
            rer_span.set_attribute("latency_ms", round(reranker_latency_ms, 2))
            rer_span.set_status(Status(StatusCode.OK))
            ctx = rer_span.get_span_context()
            reranker_span_id = format(ctx.span_id, "016x")

        with tracer.start_as_current_span("research_report_agent") as agent_span:
            agent_span.set_attribute("openinference.span.kind", "AGENT")
            agent_span.set_attribute("agent.name", "research_report_agent")
            agent_input = json.dumps({"request": "synthesize_analyst_answer", "query": query[:200], "instruction": "Retrieve -> Rerank -> Synthesize with LLM."})
            agent_span.set_attribute("input.value", agent_input)
            agent_span.set_attribute("input.mime_type", "application/json")
            if session_id:
                agent_span.set_attribute("session.id", session_id)
            agent_span.set_attribute("latency_ms", round(agent_latency_ms, 2))
            time.sleep(0.002)
            ctx = agent_span.get_span_context()
            agent_span_id = format(ctx.span_id, "016x")

            with tracer.start_as_current_span("synthesize_analyst_answer") as llm_span:
                if session_id:
                    llm_span.set_attribute("session.id", session_id)
                llm_span.set_attribute("openinference.span.kind", "LLM")
                llm_span.set_attribute("llm.model_name", VERTEX_MODEL_ID)
                llm_span.set_attribute("llm.provider", "vertex")
                llm_span.set_attribute("llm.prompt_template.template", prompt_template_str)
                llm_span.set_attribute("llm.prompt_template.version", prompt_version)
                llm_span.set_attribute("llm.prompt_template.variables", json.dumps(prompt_variables))
                llm_span.set_attribute("llm.input_messages.0.message.role", "system")
                llm_span.set_attribute("llm.input_messages.0.message.content", system_msg)
                llm_span.set_attribute("llm.input_messages.1.message.role", "user")
                llm_span.set_attribute("llm.input_messages.1.message.content", user_msg[:2000])
                llm_span.set_attribute("llm.output_messages.0.message.role", "assistant")
                llm_span.set_attribute("llm.output_messages.0.message.content", response)
                llm_span.set_attribute("llm.token_count.prompt", 450)
                llm_span.set_attribute("llm.token_count.completion", 130)
                llm_span.set_attribute("llm.token_count.total", 580)
                llm_span.set_attribute("input.value", user_msg[:2000])
                llm_span.set_attribute("output.value", response)
                llm_span.set_attribute("latency_ms", round(llm_latency_ms, 2))
                llm_span.set_status(Status(StatusCode.OK))
                ctx = llm_span.get_span_context()
                llm_span_id = format(ctx.span_id, "016x")

            agent_span.set_attribute("output.value", response)
            agent_span.set_attribute("output.mime_type", "text/plain")

        chain_span.set_attribute("output.value", response[:500])
        chain_span.set_attribute("output.mime_type", "text/plain")
        chain_span.set_attribute("latency_ms", round(chain_latency_ms, 2))
        chain_span.set_status(Status(StatusCode.OK))

    return {
        "span_data": {
            "chain_span_id": chain_span_id,
            "retrieval_span_id": retrieval_span_id,
            "reranker_span_id": reranker_span_id,
            "agent_span_id": agent_span_id,
            "llm_span_id": llm_span_id,
            "query": query,
            "response": response,
            "query_id": data["query_id"],
            "session_id": session_id,
        }
    }

In [10]:
# Session evaluations from CHAIN spans (grouped by session_id) - AnalystReportSession, realistic explanations
def generate_session_evaluations_from_chain_spans(span_df: pd.DataFrame) -> list:
    session_evaluations = []
    chain_spans = span_df[span_df["openinference.span.kind"] == "CHAIN"].copy()
    if len(chain_spans) == 0:
        return session_evaluations
    if "session_id" not in chain_spans.columns:
        return session_evaluations
    chain_spans = chain_spans[chain_spans["session_id"].notna()]
    if len(chain_spans) == 0:
        return session_evaluations
    if "timestamp" not in chain_spans.columns:
        chain_spans["timestamp"] = chain_spans.index
    chain_spans_sorted = chain_spans.sort_values("timestamp")
    oldest_per_session = chain_spans_sorted.groupby("session_id").first().reset_index()
    print(f"  Processing {len(oldest_per_session):,} unique sessions for session evaluations")
    SESSION_PASS = [
        "Full pipeline completed: retrieval returned relevant sector reports, reranker prioritized earnings and macro docs, and the model answer cited only retrieved sources with no unsourced figures.",
        "End-to-end run succeeded: document set matched the analyst question and the synthesized answer stayed within context.",
    ]
    SESSION_FAIL = [
        "Pipeline issue: retrieval may have missed key documents for the ask, or the final answer included claims not supported by the retrieved set.",
        "Session failed validation: either rerank order was suboptimal for the query or the response deviated from the provided documents.",
    ]
    for _, row in oldest_per_session.iterrows():
        span_id = row.get("context.span_id", "")
        session_id = row.get("session_id", "unknown")
        will_pass = random.random() < 0.82
        label = "pass" if will_pass else "fail"
        score = 0 if will_pass else 1
        explanations = SESSION_PASS if will_pass else SESSION_FAIL
        session_evaluations.append({
            "context.span_id": span_id,
            "session_eval.AnalystReportSession.label": label,
            "session_eval.AnalystReportSession.score": score,
            "session_eval.AnalystReportSession.explanation": random.choice(explanations),
        })
    return session_evaluations

In [11]:
# Batch collection: CHAIN, RETRIEVER, RERANKER, AGENT, LLM for evaluation generation
print("Batch cell collects CHAIN, RETRIEVER, RERANKER, AGENT, LLM for evaluation generation.")

Batch cell collects CHAIN, RETRIEVER, RERANKER, AGENT, LLM for evaluation generation.


In [12]:
# Evaluation generation: AGENT, LLM (relevance + hallucination) - realistic, domain-specific explanations
def generate_agent_trajectory_accuracy_eval(span_data: dict) -> dict:
    will_pass = random.random() < 0.75
    if will_pass:
        label, score = "pass", 0
        explanations = [
            "Agent followed the intended flow: ran document retrieval, then rerank, then invoked synthesis with the correct reranked context.",
            "Orchestration correct: retrieval then rerank then LLM sequence observed with no skipped or out-of-order steps.",
        ]
    else:
        label, score = "fail", 1
        explanations = [
            "Agent deviated from expected sequence (e.g. synthesis step called before rerank completed or retrieval was bypassed).",
            "Orchestration failed: step order or inputs did not match the required retrieval-then-rerank-then-synthesize flow.",
        ]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_hallucination_eval(span_data: dict) -> dict:
    will_pass = random.random() < 0.85
    if will_pass:
        label, score = "pass", 0
        explanations = [
            "Answer is grounded in the provided documents; no unsourced estimates, deal names, or report titles.",
            "No hallucination detected: all cited reports and figures appear in the retrieved set.",
        ]
    else:
        label, score = "fail", 1
        explanations = [
            "Response includes specific numbers, deal sizes, or report titles not present in the retrieved documents.",
            "Hallucination detected: at least one claim (e.g. consensus estimate, deal value) could not be traced to the supplied docs.",
        ]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def generate_relevance_eval(span_data: dict) -> dict:
    will_pass = random.random() < 0.88
    if will_pass:
        label, score = "pass", 0
        explanations = [
            "Answer addresses the analyst question (e.g. consensus estimates, margin comparison, M&A summary) and uses an appropriate document set.",
            "Relevant: response directly answers the ask and aligns with the intent (sector, metric, or theme).",
        ]
    else:
        label, score = "fail", 1
        explanations = [
            "Answer does not fully address the query (wrong sector, missing key ask, or off-topic).",
            "Relevance failed: response is generic or does not match the analyst's specific question (e.g. margins vs revenue, wrong region).",
        ]
    return {"label": label, "score": score, "explanation": random.choice(explanations)}

def create_evaluations_from_span_data(span_df: pd.DataFrame) -> pd.DataFrame:
    """Create evaluation DataFrame. Handles CHAIN (skip), RETRIEVER (skip), RERANKER (skip), AGENT, LLM."""
    evaluations = []
    for _, row in span_df.iterrows():
        span_kind = row.get("openinference.span.kind", "")
        span_id = row.get("context.span_id", "")
        span_data = {"query": row.get("query", ""), "response": row.get("response", "")}
        if span_kind == "AGENT":
            eval_data = generate_agent_trajectory_accuracy_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "trace_eval.AgentTrajectoryAccuracy.label": eval_data["label"],
                "trace_eval.AgentTrajectoryAccuracy.score": eval_data["score"],
                "trace_eval.AgentTrajectoryAccuracy.explanation": eval_data["explanation"],
            })
        elif span_kind == "LLM":
            hall = generate_hallucination_eval(span_data)
            rel = generate_relevance_eval(span_data)
            evaluations.append({
                "context.span_id": span_id,
                "eval.Hallucination.label": hall["label"],
                "eval.Hallucination.score": hall["score"],
                "eval.Hallucination.explanation": hall["explanation"],
                "eval.Relevance.label": rel["label"],
                "eval.Relevance.score": rel["score"],
                "eval.Relevance.explanation": rel["explanation"],
            })
    return pd.DataFrame(evaluations)

# Single Research Report Trace Test

In [13]:
# Test one trace
tracer = trace.get_tracer(__name__)
with timer():
    result = create_search_augment_trace(tracer=tracer, query_id=0, session_id="test_session_001")
    sd = result["span_data"]
    print(f"Created research report trace for query: {sd['query'][:50]}...")
    print(f"  chain_span_id: {sd['chain_span_id']}, agent_span_id: {sd['agent_span_id']}, llm_span_id: {sd['llm_span_id']}")
    print(f"  retrieval_span_id: {sd['retrieval_span_id']}, reranker_span_id: {sd['reranker_span_id']}")
    print("Created test trace – check in Arize for data.")
    flush_success = trace.get_tracer_provider().force_flush(timeout_millis=30000)
    if flush_success:
        print("Single trace exported successfully")
    else:
        print("Flush timeout - span may not have been exported")

Created research report trace for query: What's the consensus revenue estimate for AAPL in ...
  chain_span_id: 6e7e54d463bcc98f, agent_span_id: 36cc7d7cf0bf16ff, llm_span_id: a60da7fc69a71d96
  retrieval_span_id: ec0211945b308330, reranker_span_id: 5ed9b296a10a6f13
Created test trace – check in Arize for data.
Single trace exported successfully
Execution time: 0.60 seconds


# Full batch research report trace creation

In [14]:
# Batch generation: create_search_augment_trace for each run; collect CHAIN, RETRIEVER, RERANKER, AGENT, LLM
NUM_SPANS = 500
QUERY_IDS = list(range(len(SEARCH_AUGMENT_CONFIG)))
tracer = trace.get_tracer(__name__)
print(f"Generating {NUM_SPANS:,} research report traces (CHAIN -> RETRIEVER -> RERANKER -> AGENT -> LLM)...")
print("Note: BatchSpanProcessor batches automatically.\n")

with timer():
    span_data_list = []
    SESSION_SIZE_MIN = 2
    SESSION_SIZE_MAX = 5
    current_session_id = None
    traces_in_current_session = 0
    session_size = 0
    BATCH_SIZE = 50

    for batch_start in range(0, NUM_SPANS, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, NUM_SPANS)
        for i in range(batch_start, batch_end):
            if current_session_id is None or traces_in_current_session >= session_size:
                current_session_id = f"session_{random.randint(100000, 999999)}"
                session_size = random.randint(SESSION_SIZE_MIN, SESSION_SIZE_MAX)
                traces_in_current_session = 0
            query_id = random.choice(QUERY_IDS)
            result = create_search_augment_trace(tracer=tracer, query_id=query_id, session_id=current_session_id)
            traces_in_current_session += 1

            if "span_data" in result:
                sd = result["span_data"]
                span_data_list.append({"context.span_id": sd["chain_span_id"], "openinference.span.kind": "CHAIN", "query": sd["query"], "response": sd["response"], "session_id": sd.get("session_id")})
                span_data_list.append({"context.span_id": sd["retrieval_span_id"], "openinference.span.kind": "RETRIEVER", "query": sd["query"], "response": sd["response"]})
                span_data_list.append({"context.span_id": sd["reranker_span_id"], "openinference.span.kind": "RERANKER", "query": sd["query"], "response": sd["response"]})
                span_data_list.append({"context.span_id": sd["agent_span_id"], "openinference.span.kind": "AGENT", "query": sd["query"], "response": sd["response"]})
                span_data_list.append({"context.span_id": sd["llm_span_id"], "openinference.span.kind": "LLM", "query": sd["query"], "response": sd["response"]})

        if (batch_end % 500 == 0) or (batch_end == NUM_SPANS):
            print(f"  Created {batch_end:,} / {NUM_SPANS:,} traces...")

    print(f"\nCompleted creating {NUM_SPANS:,} traces. Flushing remaining spans...")
    flush_timeout = min(30000 + (NUM_SPANS // 10000) * 10000, 300000)
    flush_success = trace.get_tracer_provider().force_flush(timeout_millis=flush_timeout)
    if flush_success:
        print(f"Flush successful. Collected {len(span_data_list):,} span records for evaluation generation.")
    else:
        print("Flush timeout – some spans may not have been exported.")

collected_span_data = span_data_list

Generating 500 research report traces (CHAIN -> RETRIEVER -> RERANKER -> AGENT -> LLM)...
Note: BatchSpanProcessor batches automatically.

  Created 500 / 500 traces...

Completed creating 500 traces. Flushing remaining spans...
Flush successful. Collected 2,500 span records for evaluation generation.
Execution time: 4.31 seconds


In [15]:
# Evaluation Generation and Logging - run AFTER batch span generation
# Logs full eval_df to Arize in one call (same pattern as synthetic_spans_for_citizens.ipynb)
print("=" * 60)
print("Generating evaluations from collected span data...")
print("=" * 60 + "\n")

if "collected_span_data" not in globals() or not collected_span_data:
    print("No span data found. Please run the span generation cell first.")
else:
    span_df = pd.DataFrame(collected_span_data)
    print(f"Collected {len(span_df):,} spans")
    print(f"  - CHAIN: {len(span_df[span_df['openinference.span.kind'] == 'CHAIN']):,}")
    print(f"  - RETRIEVER: {len(span_df[span_df['openinference.span.kind'] == 'RETRIEVER']):,}")
    print(f"  - RERANKER: {len(span_df[span_df['openinference.span.kind'] == 'RERANKER']):,}")
    print(f"  - AGENT: {len(span_df[span_df['openinference.span.kind'] == 'AGENT']):,}")
    print(f"  - LLM: {len(span_df[span_df['openinference.span.kind'] == 'LLM']):,}")
    print("\nGenerating evaluations for AGENT, LLM spans...")
    eval_df = create_evaluations_from_span_data(span_df)
    print("\nGenerating session evaluations from CHAIN spans...")
    session_evaluations = generate_session_evaluations_from_chain_spans(span_df)
    if session_evaluations:
        session_eval_df = pd.DataFrame(session_evaluations)
        eval_df = pd.concat([eval_df, session_eval_df], ignore_index=True)
        print(f"  Added {len(session_evaluations):,} session evaluations")
    if len(eval_df) > 0:
        print(f"\nGenerated {len(eval_df):,} evaluations\n")
        agent_col = "trace_eval.AgentTrajectoryAccuracy.label"
        session_col = "session_eval.AnalystReportSession.label"
        agent_evals = len(eval_df[eval_df[agent_col].notna()]) if agent_col in eval_df.columns else 0
        hall_evals = len(eval_df[eval_df["eval.Hallucination.label"].notna()]) if "eval.Hallucination.label" in eval_df.columns else 0
        rel_evals = len(eval_df[eval_df["eval.Relevance.label"].notna()]) if "eval.Relevance.label" in eval_df.columns else 0
        session_evals = len(eval_df[eval_df[session_col].notna()]) if session_col in eval_df.columns else 0
        print(f"  - AgentTrajectoryAccuracy: {agent_evals:,}")
        print(f"  - Hallucination: {hall_evals:,}")
        print(f"  - Relevance: {rel_evals:,}")
        print(f"  - Session evaluations: {session_evals:,}")
        if agent_col in eval_df.columns:
            print(f"    AgentTrajectoryAccuracy: {(eval_df[agent_col] == 'pass').sum():,} pass, {(eval_df[agent_col] == 'fail').sum():,} fail")
        if "eval.Hallucination.label" in eval_df.columns:
            print(f"    Hallucination: {(eval_df['eval.Hallucination.label'] == 'pass').sum():,} pass, {(eval_df['eval.Hallucination.label'] == 'fail').sum():,} fail")
        if "eval.Relevance.label" in eval_df.columns:
            print(f"    Relevance: {(eval_df['eval.Relevance.label'] == 'pass').sum():,} pass, {(eval_df['eval.Relevance.label'] == 'fail').sum():,} fail")
        if session_col in eval_df.columns:
            print(f"    Session evaluation: {(eval_df[session_col] == 'pass').sum():,} pass, {(eval_df[session_col] == 'fail').sum():,} fail")
        print("\n" + "=" * 60)
        print("Waiting 30s for span ingestion before logging evaluations...")
        import time
        time.sleep(30)
        print("Logging evaluations to Arize...")
        print("=" * 60 + "\n")
        try:
            from arize.pandas.logger import Client
            client_kwargs = {"space_id": space_id, "api_key": api_key}
            if "endpoint" in globals() and endpoint:
                client_kwargs["URI"] = endpoint
            client = Client(**client_kwargs)
            client.log_evaluations_sync(eval_df, project_name)
            print(f"Successfully logged {len(eval_df):,} evaluations to Arize. Check the dashboard to verify.")
        except Exception as e:
            print(f"Error logging evaluations: {e}")
            print("   Evaluation DataFrame is available as 'eval_df' variable.")
            print("   You can log it manually later or fix the error and retry.")
    else:
        print("No evaluations generated. Check span data collection.")

Generating evaluations from collected span data...

Collected 2,500 spans
  - CHAIN: 500
  - RETRIEVER: 500
  - RERANKER: 500
  - AGENT: 500
  - LLM: 500

Generating evaluations for AGENT, LLM spans...

Generating session evaluations from CHAIN spans...
  Processing 141 unique sessions for session evaluations
  Added 141 session evaluations

Generated 1,141 evaluations

  - AgentTrajectoryAccuracy: 500
  - Hallucination: 500
  - Relevance: 500
  - Session evaluations: 141
    AgentTrajectoryAccuracy: 376 pass, 124 fail
    Hallucination: 427 pass, 73 fail
    Relevance: 442 pass, 58 fail
    Session evaluation: 118 pass, 23 fail

Waiting 30s for span ingestion before logging evaluations...
Logging evaluations to Arize...

  arize.utils.logging | INFO | ✅ All 1141 evaluation data have been logged successfully for model 'findata_research_report_augment-6'!
Successfully logged 1,141 evaluations to Arize. Check the dashboard to verify.
